In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split,cross_val_score, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

%matplotlib inline

In [ ]:
file_id='1Y2pakOW0OaYifT7_tHD5xa6oZnOHkmm1'
url=f'https://drive.google.com/uc?id={file_id}'
df=pd.read_csv(url).dropna(axis=1)
print(df.head())

In [ ]:
#checking whether the data is balanced or not
disease_count=df["prognosis"].value_counts()
temp_df=pd.DataFrame({
    "Disease":disease_count.index,
    "Counts":disease_count.values
})
plt.figure(figsize=(18,8))
sns.barplot(x="Disease",y="Counts",data=temp_df)
plt.xticks(rotation=90)
plt.show()

In [ ]:
encoder=LabelEncoder()
df["prognosis"]=encoder.fit_transform(df["prognosis"])

In [ ]:
X=df.drop("prognosis",axis=1)
y=df["prognosis"]
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=24)
print(f"Train:{X_train.shape},{y_train.shape}")
print(f"Test:{X_test.shape},{y_test.shape}")

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
rf_grid = GridSearchCV(rf, rf_params, cv=5, n_jobs=-1)
rf_grid.fit(X_train, y_train)
print("Best Random Forest params:", rf_grid.best_params_)


In [ ]:
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1]
}
xgb_grid = GridSearchCV(xgb, xgb_params, cv=5, n_jobs=-1)
xgb_grid.fit(X_train, y_train)
print("Best XGBoost params:", xgb_grid.best_params_)

In [ ]:
svm = SVC(probability=True)  # enabling probability for Voting Classifier compatibility
svm_params = {
    'C': [0.1, 1],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf']
}
svm_grid = GridSearchCV(svm, svm_params, cv=5, n_jobs=-1)
svm_grid.fit(X_train, y_train)
print("Best SVM params:", svm_grid.best_params_)

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_grid.best_estimator_),
        ('xgb', xgb_grid.best_estimator_),
        ('svm', svm_grid.best_estimator_)
    ],
    voting='soft'
)

voting_clf.fit(X_train, y_train)

In [ ]:
y_pred = voting_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

import joblib
joblib.dump(voting_clf, 'disease_prediction_model.pkl')

In [ ]:
def predict_disease(symptom_list):
    symptoms_index = X.columns.tolist()
    input_data = np.zeros(len(symptoms_index))
    for symptom in symptom_list:
        if symptom in symptoms_index:
            idx = symptoms_index.index(symptom)
            input_data[idx] = 1
        else:
            print(f"Warning: Symptom '{symptom}' not found in dataset.")
    prediction_encoded = voting_clf.predict([input_data])[0]
    predicted_disease = pd.factorize(y)[1][prediction_encoded]
    return predicted_disease

# example prediction:
new_symptoms = ['itching', 'skin_rash', 'nodal_skin_eruptions']
predicted_disease = predict_disease(new_symptoms)
print(f"\nPredicted Disease for symptoms {new_symptoms}: {predicted_disease}")